# Weekend- Saturday and Sunday Ridership and Stop Data Cleaning, Standardization, and Integration Pipeline

This script prepares and integrates transit ridership data with stop-level data by cleaning inconsistencies, applying agency-specific fixes, and merging datasets using appropriate matching strategies.

**Summary**
- Load & filter data: Ridership data is standardized and filtered to weekend; stop data is aligned with consistent agency names.
- Clean data: Fixes inconsistencies in route_id, stop_id, and stop_code across agencies to improve matching.
- Split by strategy: Agencies are grouped based on how they can be matched (stop_id, stop_code, or stop_name).
- Merge datasets: Each group is merged with stop data, keeping only successful matches.
- Combine results: All matched data is concatenated into a final dataset linking weekend ridership with stops and routes.

In [1]:
pip install shared_utils

ERROR: Could not find a version that satisfies the requirement shared_utils (from versions: none)
ERROR: No matching distribution found for shared_utils
Note: you may need to restart the kernel to use updated packages.


In [2]:
# Importing necessary package 
import pandas as pd 
import geopandas as gpd
import numpy as np
import google.auth
import os
import gcsfs
from calitp_data_analysis.sql import get_engine
from calitp_data_analysis import utils
db_engine = get_engine()
credentials, project = google.auth.default()
from pandas.tseries.holiday import USFederalHolidayCalendar
fs = gcsfs.GCSFileSystem()
from rapidfuzz import process, fuzz

pd.set_option('display.max_columns', None)

In [3]:
# Read in ridership data 
GCS_FILE_PATH  = 'gs://calitp-analytics-data/data-analyses/ahsc_grant/ahsc_riderships/AHSC_2026'

In [4]:
ridership_data_saturday = (
    pd.read_csv(f"{GCS_FILE_PATH}/ridership_all.csv")
    .sort_values(by='stop_name')
    .loc[lambda df: df['day_type'] == "Saturday"]
    .reset_index(drop=True)
)

ridership_data_saturday.info()
ridership_data_saturday.head(2)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 13051 entries, 0 to 13050
Data columns (total 10 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   organization_name         13051 non-null  object 
 1   stop_id                   13021 non-null  object 
 2   stop_name                 9357 non-null   object 
 3   stop_lat                  3613 non-null   float64
 4   stop_lon                  3613 non-null   float64
 5   day_type                  13051 non-null  object 
 6   average_daily_boardings   13051 non-null  float64
 7   average_daily_alightings  10963 non-null  float64
 8   start_date                13051 non-null  object 
 9   end_date                  13051 non-null  object 
dtypes: float64(4), object(6)
memory usage: 1019.7+ KB


,organization_name,stop_id,stop_name,stop_lat,stop_lon,day_type,average_daily_boardings,average_daily_alightings,start_date,end_date
0,Samtrans,345017,1000 El Camino Real-Menlo College,37.457839,-122.190513,Saturday,9.4,16.2,2025-08-01,2025-08-31
1,Golden Gate Transit,40469,1011 Andersen Dr,NaN,NaN,Saturday,3.5,0.0,2025-09-01,2025-09-30


In [5]:
ridership_data_sunday = (
    pd.read_csv(f"{GCS_FILE_PATH}/ridership_all.csv")
    .sort_values(by='stop_name')
    .loc[lambda df: df['day_type'] == "Sunday"]
    .reset_index(drop=True)
)

ridership_data_sunday.info()
ridership_data_sunday.head(2)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12526 entries, 0 to 12525
Data columns (total 10 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   organization_name         12526 non-null  object 
 1   stop_id                   12496 non-null  object 
 2   stop_name                 8879 non-null   object 
 3   stop_lat                  3528 non-null   float64
 4   stop_lon                  3528 non-null   float64
 5   day_type                  12526 non-null  object 
 6   average_daily_boardings   12526 non-null  float64
 7   average_daily_alightings  10443 non-null  float64
 8   start_date                12526 non-null  object 
 9   end_date                  12526 non-null  object 
dtypes: float64(4), object(6)
memory usage: 978.7+ KB


,organization_name,stop_id,stop_name,stop_lat,stop_lon,day_type,average_daily_boardings,average_daily_alightings,start_date,end_date
0,Samtrans,345017,1000 El Camino Real-Menlo College,37.457839,-122.190513,Sunday,8.8,15.6,2025-08-01,2025-08-31
1,Golden Gate Transit,40469,1011 Andersen Dr,NaN,NaN,Sunday,1.5,0.0,2025-09-01,2025-09-30


In [6]:
agency_mapping = {
    'Gold Coast Schedule': 'Gold Coast Transit',
    'Burbank Schedule': 'City of Burbank',
    'SamTrans Schedule': 'Samtrans',
    'Big Blue Bus Schedule': 'Big Blue Bus',
    'Foothill Schedule': 'Foothill Transit',
    'San Diego Schedule': 'SDMTS',
    'Golden Gate Park Shuttle Schedule': 'Golden Gate Park Shuttle',
    'Fresno Schedule': 'Fresno County',
    'OmniTrans Schedule': 'Omnitrans',
    'Sacramento Schedule': 'SacRT Bus',
    'BART Schedule': 'San Francisco Bay Area Rapid Transit District',
    'Riverside Schedule': 'Riverside Transit',
    'OCTA Schedule': 'Orange County Transportation Authority',
    'Santa Cruz Schedule': 'Santa Cruz Metro',
    'Long Beach Schedule': 'Long Beach Transit',
    'Caltrain Schedule': 'Caltrain',
    'SBMTD Schedule': 'SBMTD',
    'Culver City Schedule': 'Culver City Bus',
    'Golden Gate Transit Schedule': 'Golden Gate Transit'
}

In [7]:
stops_aggregated_saturday = (
    pd.read_csv(f"{GCS_FILE_PATH}/stops_aggregated_saturday.csv")
    .assign(organization_name=lambda df: df['name'].map(agency_mapping))
)

stops_aggregated_sunday = (
    pd.read_csv(f"{GCS_FILE_PATH}/stops_aggregated_sunday.csv")
    .assign(organization_name=lambda df: df['name'].map(agency_mapping))
)

stops_aggregated_saturday.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 23157 entries, 0 to 23156
Data columns (total 9 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   feed_key           23157 non-null  object
 1   stop_id            23157 non-null  object
 2   stop_name          23157 non-null  object
 3   n_arrivals         23157 non-null  int64 
 4   n_routes           23157 non-null  int64 
 5   stop_code          20417 non-null  object
 6   pt_geom            23157 non-null  object
 7   name               23157 non-null  object
 8   organization_name  23157 non-null  object
dtypes: int64(2), object(7)
memory usage: 1.6+ MB


In [8]:
# Upon checking, found that Some stops are unmatched because Big Blue Bus ridership uses negative stop_codes.
# These correspond to valid stop_codes in the stop data (same route and stop_name).
# The mapping below replaces negative ridership stop_codes with the correct stop_codes
# so they can match during the merge.

for ridership_df, stops_df in [
    (ridership_data_saturday, stops_aggregated_saturday),
    (ridership_data_sunday, stops_aggregated_sunday),
]:

    # --- Big Blue Bus stop_id mapping ---
    mask_bb = ridership_df['organization_name'] == 'Big Blue Bus'

    ridership_df.loc[mask_bb, 'stop_id'] = (
        ridership_df.loc[mask_bb, 'stop_id']
        .astype(int)
        .replace({
            -4: 'MNSWSMNF',
            -5: '014MCHNN',
            -6: '014PCONF',
            -7: '014PCOSF',
            -8: '017PRLNN',
            -9: '017PRLSF',
            -13: 'PRL014EF',
            -14: 'PRL014WN',
            -18: 'SMCBUNPL'
        })
        .astype(str)
    )

    #  clean stop_code
    stops_df.loc[
        stops_df['organization_name'] == 'Big Blue Bus',
        'stop_code'
    ] = (
        stops_df.loc[
            stops_df['organization_name'] == 'Big Blue Bus',
            'stop_code'
        ]
        .astype(str)
        .str.strip()
    )

# Upon checking, found that Gold Coast Schedule stop_ids have some issues:
# 1. Some stop_ids contain ':' which prevents correct merge with ridership data
# 2. Two specific stop_ids ('GCTDMAIN1' and 'SAClMAI1') do not match the ridership data stop_ids ('GCTDMAIN' and 'SACLMAI1')
# So we remove colons and update these two stop_ids before merging
for df in [stops_aggregated_saturday, stops_aggregated_sunday]:
    mask = df['name'] == "Gold Coast Schedule"
    
    df.loc[mask, 'stop_id'] = (
        df.loc[mask, 'stop_id']
        .str.replace(':', '', regex=False)
        .replace({
            'GCTDMAIN1': 'GCTDMAIN',
            'SAClMAI1': 'SACLMAI1'
        })
    )


# Upon checking found that Long Beach transit stop_id has leading zeros like 0110, 0220 and so on
agencies = [
    'Long Beach Transit', 'Orange County Transportation Authority'
]

for df in [stops_aggregated_saturday, stops_aggregated_sunday]:
    for agency in agencies:
        mask = df['organization_name'] == agency

        df.loc[mask, 'stop_id'] = (
            df.loc[mask, 'stop_id']
            .astype(str)
            .str.replace(r'^0+', '', regex=True)
        )


In [9]:
master_cols = [
    "organization_name",
    "feed_key",
    "stop_id",
    "stop_name",
    "stop_code",
    "n_arrivals",
    "n_routes",
    "pt_geom",
    "day_type",
    "average_daily_boardings",
    "average_daily_alightings",
    "start_date",
    "end_date"
]

def standardize_columns(df, master_cols):
    
    # add missing columns
    for col in master_cols:
        if col not in df.columns:
            df[col] = None

    # keep only master columns and order them
    return df[master_cols]

### Agencies with Saturday and Sunday Data, Categorized by Match Type: route*stop_id, stop_code, or stop_name

In [10]:
ridership_data_sunday.organization_name.unique()

array(['Samtrans', 'Golden Gate Transit', 'Long Beach Transit', 'SDMTS',
       'Golden Gate Park Shuttle', 'Fresno County',
       'San Francisco Bay Area Rapid Transit District', 'Big Blue Bus',
       'Caltrain', 'Culver City Bus', 'Foothill Transit',
       'Riverside Transit'], dtype=object)

In [11]:
agencies__with_stop_id_match = [
    'Foothill Transit',
    # 'Gold Coast Transit',
    'Golden Gate Transit',
    'Long Beach Transit',
    # 'Orange County Transportation Authority',
    # 'Omnitrans',    
    # 'SacRT Bus',
    'Samtrans',
    'SDMTS',
    'San Francisco Bay Area Rapid Transit District', 
    'Fresno County',
    # 'SBMTD'
]

agencies_with_stop_code_match = [
    'Culver City Bus',
    'Big Blue Bus',
    'Riverside Transit',
    # 'Santa Cruz Metro',
    'Golden Gate Park Shuttle',
]



agencies_with_stop_name_match = [
                          'Caltrain',
                          # 'City of Burbank',
                         ]

In [12]:
# Split Ridership by categories

def split_ridership_by_agency(ridership_df, agency_groups):
    subsets = {}
    for name, agencies in agency_groups.items():
        subsets[name] = ridership_df[ridership_df['organization_name'].isin(agencies)]
    return subsets

agency_groups = {
    'with_stop_id_match': agencies__with_stop_id_match,
    'with_stop_code_match': agencies_with_stop_code_match,
    'with_stop_name_match': agencies_with_stop_name_match
}

ridership_subsets_saturday = split_ridership_by_agency(ridership_data_saturday, agency_groups)
ridership_subsets_sunday = split_ridership_by_agency(ridership_data_sunday, agency_groups)

In [13]:
# Merge without route_id
merge_with_stop_id_match_saturday = (
    pd.merge(
        ridership_subsets_saturday['with_stop_id_match'], 
        stops_aggregated_saturday,
        on=['organization_name', 'stop_id'],
        how='left',
        indicator=True,
        suffixes=('_x','_y')
    )
    .rename(columns={
        'stop_name_y': 'stop_name',
    })
)

# Keep only rows where the merge matched in both DataFrames
merge_with_stop_id_matched_saturday = merge_with_stop_id_match_saturday[merge_with_stop_id_match_saturday['_merge'] == 'both'].copy()

merge_counts_saturday = merge_with_stop_id_matched_saturday['_merge'].value_counts()
print(merge_counts_saturday)

_merge
both          9867
left_only        0
right_only       0
Name: count, dtype: int64


In [14]:
merge_with_stop_id_matched_saturday = standardize_columns(merge_with_stop_id_matched_saturday, master_cols)
merge_with_stop_id_matched_saturday.sample(5)

,organization_name,feed_key,stop_id,stop_name,stop_code,n_arrivals,n_routes,pt_geom,day_type,average_daily_boardings,average_daily_alightings,start_date,end_date
2692,Samtrans,db97cc02836aa5f0cf647d80160b23ec,341160,El Camino Real & Warren Rd,341160,68.0,2.0,POINT(-122.340884 37.573359),Saturday,14.888889,7.777778,2025-08-01,2025-08-31
1398,Long Beach Transit,cddd375786d835389a7beb9632369907,1412,Broadway & Molino NW,1412,31.0,2.0,POINT(-118.160887 33.765977),Saturday,8.589097,0.000000,2024-07-01,2025-06-30
8206,Long Beach Transit,cddd375786d835389a7beb9632369907,1236,Willow & Long Beach Blvd SE,1236,37.0,3.0,POINT(-118.189054 33.804312),Saturday,170.581097,67.080354,2024-07-01,2025-06-30
2342,SDMTS,1fff52f9349da228c56eef492df5001b,30675,E Palomar St & Davies Dr,30675,44.0,2.0,POINT(-117.02353235 32.61424315),Saturday,4.754871,7.065767,2024-09-01,2025-01-25
1940,Long Beach Transit,cddd375786d835389a7beb9632369907,1432,Clark & Pageantry NE,1432,15.0,1.0,POINT(-118.133915 33.813385),Saturday,0.000000,2.795276,2024-07-01,2025-06-30


In [15]:
# Merge without route_id
merge_with_stop_id_match_sunday = (
    pd.merge(
        ridership_subsets_sunday['with_stop_id_match'], 
        stops_aggregated_sunday,
        on=['organization_name', 'stop_id'],
        how='left',
        indicator=True,
        suffixes=('_x','_y')
    )
    .rename(columns={
        'stop_name_y': 'stop_name',
    })
)

# Keep only rows where the merge matched in both DataFrames
merge_with_stop_id_matched_sunday = merge_with_stop_id_match_sunday[merge_with_stop_id_match_sunday['_merge'] == 'both'].copy()

merge_counts_sunday = merge_with_stop_id_matched_sunday['_merge'].value_counts()
print(merge_counts_sunday)



_merge
both          9360
left_only        0
right_only       0
Name: count, dtype: int64


In [16]:
merge_with_stop_id_matched_sunday = standardize_columns(merge_with_stop_id_matched_sunday, master_cols)
merge_with_stop_id_matched_sunday.sample(5)

,organization_name,feed_key,stop_id,stop_name,stop_code,n_arrivals,n_routes,pt_geom,day_type,average_daily_boardings,average_daily_alightings,start_date,end_date
2616,Fresno County,23d1893801eefadf7544a670a3bcd312,65,FRESNO YOSEMITE INTL - WB,65,46.0,2.0,POINT(-119.719794 36.76923),Sunday,6.661081,5.699753,2024-09-01,2025-08-31
8513,Foothill Transit,661ef844bdaa253e8b950740f76061b1,1577,Golden Springs Dr and Racquet Club Dr S,1577,70.0,1.0,POINT(-117.818375 34.008648),Sunday,6.134615,1.750000,2024-07-01,2025-06-30
4302,Fresno County,23d1893801eefadf7544a670a3bcd312,2347,NE BRAWLEY - CLINTON,2347,24.0,1.0,POINT(-119.862145 36.772174),Sunday,4.579600,10.989460,2024-09-01,2025-08-31
9174,Foothill Transit,661ef844bdaa253e8b950740f76061b1,2950,Cogswell Rd and Lambert Ave W,2950,26.0,1.0,POINT(-118.010985 34.079321),Sunday,3.170732,0.219512,2024-07-01,2025-06-30
6861,Long Beach Transit,cddd375786d835389a7beb9632369907,87,Santa Fe & 23rd NE,0087,49.0,2.0,POINT(-118.215315 33.799215),Sunday,48.675916,35.179441,2024-07-01,2025-06-30


In [17]:
merge_with_stop_code_match_saturday = (
    pd.merge(
        ridership_subsets_saturday['with_stop_code_match'],  
        stops_aggregated_saturday,
        left_on=['organization_name', 'stop_id'],  # ridership stop_id
        right_on=['organization_name', 'stop_code'],  # stops table stop_code
        how='left',
        indicator=True
    )
    .rename(columns={
        'stop_id_y': 'stop_id',
        'stop_name_y': 'stop_name',
    })
)


# Keep only rows where the merge matched in both DataFrames
merge_with_stop_code_matched_saturday = merge_with_stop_code_match_saturday[merge_with_stop_code_match_saturday['_merge'] == 'both'].copy()

# Check merge results
merge_counts_saturday = merge_with_stop_code_matched_saturday['_merge'].value_counts()
print(merge_counts_saturday)

_merge
both          2933
left_only        0
right_only       0
Name: count, dtype: int64


In [18]:
merge_with_stop_code_matched_saturday = standardize_columns(merge_with_stop_code_matched_saturday, master_cols)
merge_with_stop_code_matched_saturday.sample(5)

,organization_name,feed_key,stop_id,stop_name,stop_code,n_arrivals,n_routes,pt_geom,day_type,average_daily_boardings,average_daily_alightings,start_date,end_date
2231,Riverside Transit,6eb2b575bee157dace7a2c7155d3cb25,1346,Alta Murrieta + Brownestone,2490,20.0,1.0,POINT(-117.169753 33.564895),Saturday,1.571429,NaN,2025-01-01,2025-10-31
262,Culver City Bus,f6774d861953d4f4cdcffec95e2652c7,712,Kinross Ave/Veteran Ave,909,66.0,1.0,POINT(-118.447925 34.059517),Saturday,54.500000,2.7,2025-07-14,2025-08-25
254,Culver City Bus,f6774d861953d4f4cdcffec95e2652c7,469,Jefferson Blvd/Ballona Ln,440,45.0,2.0,POINT(-118.395673 33.998464),Saturday,1.000000,2.7,2025-07-14,2025-08-25
2751,Riverside Transit,6eb2b575bee157dace7a2c7155d3cb25,2352,San Jacinto + Esplanade,3367,16.0,1.0,POINT(-116.958691 33.773419),Saturday,4.424242,NaN,2025-01-01,2025-10-31
2030,Riverside Transit,6eb2b575bee157dace7a2c7155d3cb25,1063,Limonite + Baldwin,2222,23.0,2.0,POINT(-117.467472 33.975816),Saturday,2.615385,NaN,2025-01-01,2025-10-31


In [19]:
merge_with_stop_code_match_sunday = (
    pd.merge(
        ridership_subsets_sunday['with_stop_code_match'],  
        stops_aggregated_sunday,
        left_on=['organization_name', 'stop_id'],  # ridership stop_id
        right_on=['organization_name', 'stop_code'],  # stops table stop_code
        how='left',
        indicator=True
    )
    .rename(columns={
        'stop_id_y': 'stop_id',
        'stop_name_y': 'stop_name',
    })
)


# Keep only rows where the merge matched in both DataFrames
merge_with_stop_code_matched_sunday = merge_with_stop_code_match_sunday[merge_with_stop_code_match_sunday['_merge'] == 'both'].copy()

# Check merge results
merge_counts_sunday = merge_with_stop_code_matched_sunday['_merge'].value_counts()
print(merge_counts_sunday)

_merge
both          2931
left_only        0
right_only       0
Name: count, dtype: int64


In [20]:
merge_with_stop_code_matched_sunday = standardize_columns(merge_with_stop_code_matched_sunday, master_cols)
merge_with_stop_code_matched_sunday.sample(5)

,organization_name,feed_key,stop_id,stop_name,stop_code,n_arrivals,n_routes,pt_geom,day_type,average_daily_boardings,average_daily_alightings,start_date,end_date
2019,Riverside Transit,6eb2b575bee157dace7a2c7155d3cb25,1057,Van Buren + Central,2216,10.0,1.0,POINT(-117.457745 33.956212),Sunday,1.000000,NaN,2025-01-01,2025-10-31
2880,Riverside Transit,6eb2b575bee157dace7a2c7155d3cb25,1157,Tustin + Chestnut,3623,9.0,1.0,POINT(-117.835954 33.813054),Sunday,10.611111,NaN,2025-01-01,2025-10-31
2569,Riverside Transit,6eb2b575bee157dace7a2c7155d3cb25,1833,Gilman Springs + Soboba,2932,11.0,1.0,POINT(-116.973719 33.819547),Sunday,1.000000,NaN,2025-01-01,2025-10-31
870,Big Blue Bus,7a3f513c343b16a30c135ed7d332b6d6,539,WILSHIRE BLVD & BROCKTON AVE,2172,30.0,1.0,POINT(-118.465271 34.045761),Sunday,9.122823,15.501751,2024-08-01,2025-11-30
1719,Riverside Transit,6eb2b575bee157dace7a2c7155d3cb25,661,La Sierra + Norwood,1856,17.0,1.0,POINT(-117.491593 33.923631),Sunday,5.333333,NaN,2025-01-01,2025-10-31


In [21]:
cols_to_keep = [
    'feed_key', 'stop_id',
    'n_arrivals', 'n_routes', 'pt_geom'
]

def fuzzy_match_subset(left_df, right_df, threshold=80):
    results = []

    for org in left_df['organization_name'].unique():
        left_group = left_df[left_df['organization_name'] == org]
        right_group = right_df[right_df['organization_name'] == org]

        if right_group.empty:
            continue

        choices = right_group['stop_name'].tolist()

        for idx, row in left_group.iterrows():
            match, score, match_idx = process.extractOne(
                row['stop_name'],
                choices,
                scorer=fuzz.token_set_ratio
            )

            if score >= threshold:
                matched_row = right_group.iloc[match_idx]
                subset = matched_row[cols_to_keep].to_dict()
                matched_name = matched_row['stop_name']
            else:
                subset = {col: None for col in cols_to_keep}
                matched_name = None

            subset.update({
                'left_index': idx,
                'original_stop_name': row['stop_name'],
                'matched_stop_name': matched_name,
                'score': score
            })

            results.append(subset)

    return pd.DataFrame(results)


In [22]:
matches_df_saturday = fuzzy_match_subset(
    ridership_subsets_saturday['with_stop_name_match'], 
    stops_aggregated_saturday,
    threshold=80
)

In [23]:
matches_df_sunday = fuzzy_match_subset(
    ridership_subsets_sunday['with_stop_name_match'], 
    stops_aggregated_sunday,
    threshold=80
)

In [24]:
merge_with_stop_name_match_saturday = (
    ridership_subsets_saturday['with_stop_name_match']
    .merge(
        matches_df_saturday,
        left_index=True,
        right_on='left_index',
        how='left',
        indicator=True
    )
    .rename(columns={
        'stop_id_y': 'stop_id',
    })
)

# Keep only rows where the merge matched in both DataFrames
merge_with_stop_name_matched_saturday = merge_with_stop_name_match_saturday[merge_with_stop_name_match_saturday['_merge'] == 'both'].copy()

merge_counts_saturday = merge_with_stop_name_matched_saturday['_merge'].value_counts()
print(merge_counts_saturday)

_merge
both          30
left_only      0
right_only     0
Name: count, dtype: int64


In [25]:
merge_with_stop_name_matched_saturday = standardize_columns(merge_with_stop_name_matched_saturday, master_cols)
merge_with_stop_name_matched_saturday.sample(5)

,organization_name,feed_key,stop_id,stop_name,stop_code,n_arrivals,n_routes,pt_geom,day_type,average_daily_boardings,average_daily_alightings,start_date,end_date
18,Caltrain,f189d5677d4a106b98585f3c5d4fd42c,70141,Redwood City,None,33.0,1.0,POINT(-122.231936 37.486159),Saturday,799.444990,NaN,2023-11-01,2025-07-31
29,Caltrain,f189d5677d4a106b98585f3c5d4fd42c,70271,Tamien,None,17.0,1.0,POINT(-121.883721 37.31174),Saturday,58.297552,NaN,2023-11-01,2025-07-31
11,Caltrain,f189d5677d4a106b98585f3c5d4fd42c,70111,Hillsdale,None,33.0,1.0,POINT(-122.301496637 37.542607659),Saturday,486.553577,NaN,2023-11-01,2025-07-31
17,Caltrain,f189d5677d4a106b98585f3c5d4fd42c,70171,Palo Alto,None,33.0,1.0,POINT(-122.164614 37.443475),Saturday,1220.752048,NaN,2023-11-01,2025-07-31
3,Caltrain,None,None,Blossom Hill,None,NaN,NaN,None,Saturday,0.199045,NaN,2023-11-01,2025-07-31


In [26]:
merge_with_stop_name_match_sunday = (
    ridership_subsets_sunday['with_stop_name_match']
    .merge(
        matches_df_sunday,
        left_index=True,
        right_on='left_index',
        how='left',
        indicator=True
    )
    .rename(columns={
        'stop_id_y': 'stop_id',
    })
)

# Keep only rows where the merge matched in both DataFrames
merge_with_stop_name_matched_sunday = merge_with_stop_name_match_sunday[merge_with_stop_name_match_sunday['_merge'] == 'both'].copy()

merge_counts_sunday = merge_with_stop_name_matched_sunday['_merge'].value_counts()
print(merge_counts_sunday)

_merge
both          30
left_only      0
right_only     0
Name: count, dtype: int64


In [27]:
merge_with_stop_name_matched_sunday = standardize_columns(merge_with_stop_name_matched_sunday, master_cols)
merge_with_stop_name_matched_sunday.sample(5)

,organization_name,feed_key,stop_id,stop_name,stop_code,n_arrivals,n_routes,pt_geom,day_type,average_daily_boardings,average_daily_alightings,start_date,end_date
16,Caltrain,f189d5677d4a106b98585f3c5d4fd42c,70211,Mountain View,None,33.0,1.0,POINT(-122.075956 37.394459),Sunday,784.332601,NaN,2023-11-01,2025-07-31
7,Caltrain,None,None,Capitol,None,NaN,NaN,None,Sunday,0.000000,NaN,2023-11-01,2025-07-31
25,Caltrain,f189d5677d4a106b98585f3c5d4fd42c,70091,San Mateo,None,33.0,1.0,POINT(-122.323851 37.568087),Sunday,478.543508,NaN,2023-11-01,2025-07-31
4,Caltrain,f189d5677d4a106b98585f3c5d4fd42c,70071,Broadway,None,33.0,1.0,POINT(-122.36265 37.58764),Sunday,66.493661,NaN,2023-11-01,2025-07-31
24,Caltrain,None,None,San Martin,None,NaN,NaN,None,Sunday,0.000000,NaN,2023-11-01,2025-07-31


In [28]:
all_ridership_stop_trips_saturday_data = pd.concat([
    merge_with_stop_id_matched_saturday,
    merge_with_stop_code_matched_saturday,
    merge_with_stop_name_matched_saturday    
], ignore_index=True)

In [29]:
all_ridership_stop_trips_sunday_data = pd.concat([
    merge_with_stop_id_matched_sunday,
    merge_with_stop_code_matched_sunday,
    merge_with_stop_name_matched_sunday    
], ignore_index=True)

In [30]:
all_ridership_stop_trips_saturday_data.to_csv(f"{GCS_FILE_PATH}/ridership_trips_routes_saturday.csv", index=False)
all_ridership_stop_trips_sunday_data.to_csv(f"{GCS_FILE_PATH}/ridership_trips_routes_saturday.csv", index=False)